# HEILO Training no Colab (GPU grátis)

Este notebook faz duas coisas, e as duas são **opcionais**:

1. **HEILO Teacher → exemplos.** O Teacher (Qwen2.5-0.5B-Instruct, licença Apache-2.0) responde às perguntas de `data/teacher/prompts.txt`. As respostas vão para `data/teacher/generated.jsonl` como **NÃO verificadas**. Elas só entram no treino depois que você aprovar com `/revisar` no seu PC.
2. **HEILO Seed → treino.** Treina o modelo próprio da HEILO (do zero) com a última versão do dataset HEILO, que só contém exemplos aprovados. Nenhum peso do Teacher é copiado para o Seed.

**Antes de começar**

1. `Ambiente de execução → Alterar tipo → GPU (T4)`.
2. Crie um token do GitHub (fine-grained, com **Contents: Read and write** no repositório `HELIO`) e salve na 🔑 (Secrets) do Colab com o nome `GITHUB_TOKEN`. O token nunca vai para o código.
3. No seu PC, rode `python -m heilo.main aprender`, depois `/revisar`, depois `python -m heilo.main publicar`.

Quando terminar, rode `python -m heilo.main atualizar` no seu PC e depois `/revisar` para avaliar o que o Teacher gerou.

In [ ]:
REPO = "hugolindo12/HELIO"
BRANCH = "main"
GERAR_COM_TEACHER = True   # passo 1
TREINAR_SEED = True        # passo 2
PASSOS_SEED = 6000

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (ative a GPU!)")

In [ ]:
from google.colab import userdata
import os, subprocess, sys
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/heilo", "pull"], check=True)
sys.path.insert(0, "/content")

In [ ]:
!pip -q install -U transformers accelerate

## 1. HEILO Teacher gera exemplos (NÃO verificados)

In [ ]:
from heilo.config import config
from heilo.core.model_manager import ModelManager
from heilo.training.pipeline import TrainingPipeline, TeacherRequired
from pathlib import Path

models = ModelManager.from_config(config)
pipe = TrainingPipeline(models=models)
if GERAR_COM_TEACHER:
    prompts_file = Path("/content/heilo/data/teacher/prompts.txt")
    if not prompts_file.exists():
        print("Crie data/teacher/prompts.txt (uma pergunta por linha) no seu PC e publique.")
    else:
        print(models.get("teacher").card().to_dict())
        try:
            r = pipe.generate_with_teacher(prompts_file.read_text(encoding="utf-8").splitlines(), log=print)
            print(r, "→ revise no PC com /revisar")
        except TeacherRequired as e:
            print(e)

## 2. Treinar o HEILO Seed (modelo próprio, do zero)

In [ ]:
if TREINAR_SEED:
    manifest = pipe.build_dataset()        # só exemplos APROVADOS
    print(manifest)
    run = pipe.train_seed(passos=PASSOS_SEED, lote=64, log_cada=500)
    print(run)
    for p in ["oi", "quem é você?", "to cansado", "conta uma piada"]:
        print("Você:", p, "\nHEILO Seed:", models.generate_with("seed", [{"role": "user", "content": p}]).text, "\n")

In [ ]:
!cd /content/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd /content/heilo && git add data/teacher data/training models/seed/weights && git commit -m "heilo: Colab (Teacher gerou exemplos / Seed treinado)" || echo "nada novo"
!cd /content/heilo && git pull --rebase && git push
print("Pronto! No PC: python -m heilo.main atualizar  →  /revisar")